# Data Mining Assignment: Spark Data Lake
**Name:** Abhay Prasad  
**Roll number:** LIT2023006  
**Dataset:** `sales.csv` from the class repository  
**Environment:** Google Colab + PySpark

This notebook preserves the source file in **Bronze**, applies basic preprocessing using
Spark DataFrame operations, and saves and verifies **Silver**. No Gold layer or detailed
analysis is included. All discrepancy counts and row counts are computed at runtime.

In Colab select **Runtime > Run all** with a Python CPU runtime. The notebook retrieves
the original dataset automatically; no upload, Drive mount, or secret is needed.


## 1. Install PySpark and start a SparkSession


In [1]:
import importlib.util
import subprocess
import sys

if importlib.util.find_spec("pyspark") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pyspark==3.5.8"])

from pathlib import Path
import hashlib
import json
import shutil
import urllib.request
import pyspark
from pyspark.sql import SparkSession, Window
from pyspark.sql import functions as F

spark = (SparkSession.builder
         .appName("LIT2023006_Spark_Data_Lake")
         .master("local[2]")
         .config("spark.sql.shuffle.partitions", "2")
         .config("spark.sql.ansi.enabled", "false")
         .config("spark.sql.legacy.timeParserPolicy", "CORRECTED")
         .config("spark.sql.session.timeZone", "UTC")
         .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")
print("Spark version:", spark.version)


Spark version: 4.0.4


## 2. Create the lake and preserve Bronze

The source URL is pinned to a class-repository commit so reruns use the same input.
A local copy is used when running from the submission folder. The raw bytes are copied
without changing headers, values, quoting, or line endings. SHA-256 verifies that Bronze
is identical to the provided dataset before and after preprocessing.

```text
data_lake/
├── bronze/sales.csv
└── silver/preprocessed_sales/   # Spark CSV output directory
silver/sales_silver.csv          # complete single-file submission copy
```


In [2]:
SOURCE_URL = "https://raw.githubusercontent.com/necromancersanju-netizen/data-mining-data-lake-assignment/81539d8aee71b8ce4c428365cea26e6b272723c6/dataset/sales.csv"
EXPECTED_SOURCE_SHA256 = "1b2c7e0acea6250c992b890975ed0947b5bececa6a2880b707e4523e0ee22eb8"
BASE = Path.cwd()
LAKE = BASE / "data_lake"
BRONZE = LAKE / "bronze" / "sales.csv"
SILVER = LAKE / "silver" / "preprocessed_sales"
SUBMISSION_CSV = BASE / "silver" / "sales_silver.csv"
BRONZE.parent.mkdir(parents=True, exist_ok=True)
SILVER.parent.mkdir(parents=True, exist_ok=True)
SUBMISSION_CSV.parent.mkdir(parents=True, exist_ok=True)

local_candidates = [BASE / "bronze/sales.csv", BASE / "dataset/sales.csv",
                    BASE.parent.parent / "dataset/sales.csv"]
local_source = next((p for p in local_candidates if p.is_file()), None)
if local_source is not None:
    source_bytes = local_source.read_bytes()
else:
    with urllib.request.urlopen(SOURCE_URL, timeout=60) as response:
        source_bytes = response.read()

source_hash = hashlib.sha256(source_bytes).hexdigest()
assert source_hash == EXPECTED_SOURCE_SHA256, "Source differs from the assigned dataset."
if BRONZE.exists():
    assert BRONZE.read_bytes() == source_bytes, "Existing Bronze differs; it was not overwritten."
else:
    BRONZE.write_bytes(source_bytes)
print("Bronze SHA-256:", source_hash)
print("Bronze is byte-for-byte identical to the provided CSV.")


Bronze SHA-256: 1b2c7e0acea6250c992b890975ed0947b5bececa6a2880b707e4523e0ee22eb8
Bronze is byte-for-byte identical to the provided CSV.


## 3. Load and inspect using Spark

Read all source columns as strings initially. This preserves malformed numbers and dates
for inspection instead of losing them during schema inference. Empty CSV fields are null.


In [3]:
raw = (spark.read.option("header", True).option("inferSchema", False)
       .option("mode", "FAILFAST").csv(str(BRONZE.resolve())))
expected_columns = ["order_id", "customer_id", "customer_name", "product", "category",
                    "quantity", "unit_price", "discount_percent", "payment_method",
                    "city", "state", "order_date", "order_status"]
assert raw.columns == expected_columns, "Unexpected source columns."
raw.cache()
before_count = raw.count()
raw.printSchema()
raw.show(8, truncate=False)
print("Total records before preprocessing:", before_count)


root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- product: string (nullable = true)
 |-- category: string (nullable = true)
 |-- quantity: string (nullable = true)
 |-- unit_price: string (nullable = true)
 |-- discount_percent: string (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- order_date: string (nullable = true)
 |-- order_status: string (nullable = true)

+--------+-----------+---------------+-------------+---------+--------+----------+----------------+----------------+----------+--------------+----------+------------+
|order_id|customer_id|customer_name  |product      |category |quantity|unit_price|discount_percent|payment_method  |city      |state         |order_date|order_status|
+--------+-----------+---------------+-------------+---------+--------+----------+----------------+----------------+--

## 4. Identify discrepancies using Spark operations


In [4]:
# Exact duplicate means equality across every original column, not just the order ID.
deduplicated = raw.dropDuplicates().cache()
after_dedup_count = deduplicated.count()
duplicate_count = before_count - after_dedup_count
print("Exact duplicate rows to remove:", duplicate_count)
duplicate_groups = raw.groupBy(*raw.columns).count().filter(F.col("count") > 1)
duplicate_groups.select("order_id", "customer_id", "count").orderBy("order_id").show(10)

def missing(column):
    return column.isNull() | (F.trim(column) == "")

def count_true(condition):
    return F.sum(F.when(condition, 1).otherwise(0)).cast("long")

print("Null or blank values per column (original Bronze rows):")
missing_counts = raw.agg(*[count_true(missing(F.col(c))).alias(c) for c in raw.columns])
missing_counts.show(truncate=False, vertical=True)


Exact duplicate rows to remove: 10
+--------+-----------+-----+
|order_id|customer_id|count|
+--------+-----------+-----+
|  100055|      C0270|    2|
|  100145|      C0007|    2|
|  100297|      C0149|    2|
|  100317|      C0264|    2|
|  100329|      C0151|    2|
|  100444|      C0090|    2|
|  100502|      C0068|    2|
|  100588|      C0228|    2|
|  100692|      C0141|    2|
|  100773|      C0315|    2|
+--------+-----------+-----+

Null or blank values per column (original Bronze rows):
-RECORD 0---------------
 order_id         | 0   
 customer_id      | 0   
 customer_name    | 10  
 product          | 0   
 category         | 0   
 quantity         | 0   
 unit_price       | 0   
 discount_percent | 0   
 payment_method   | 8   
 city             | 8   
 state            | 0   
 order_date       | 0   
 order_status     | 0   



In [5]:
# Trim every field and represent blank/whitespace-only fields consistently as null.
def trim_and_null(df):
    return df.select(*[
        F.when(F.trim(F.col(c)) == "", F.lit(None).cast("string"))
         .otherwise(F.trim(F.col(c))).alias(c)
        for c in df.columns
    ])

trimmed_raw = trim_and_null(raw)
title_columns = ["customer_name", "category", "city", "state", "order_status"]
modal_columns = ["product", "payment_method"]
text_columns = ["order_id", "customer_id"] + title_columns + modal_columns

print("Rows with leading/trailing whitespace by text column:")
raw.agg(*[count_true(F.col(c) != F.trim(F.col(c))).alias(c)
          for c in text_columns]).show(truncate=False, vertical=True)
print("Rows requiring obvious title-case normalization:")
trimmed_raw.agg(*[count_true(F.col(c) != F.initcap(F.col(c))).alias(c)
                  for c in title_columns]).show(truncate=False, vertical=True)

# Count case variants without altering product acronyms or units such as TV and 500g.
print("Text groups with multiple capitalizations:")
for c in title_columns + modal_columns:
    variants = (trimmed_raw.filter(F.col(c).isNotNull())
                .groupBy(F.lower(F.col(c)).alias("case_insensitive_value"))
                .agg(F.countDistinct(c).alias("variant_count"),
                     F.sort_array(F.collect_set(c)).alias("observed_variants"))
                .filter(F.col("variant_count") > 1))
    print(c, "- inconsistent groups:", variants.count())
    variants.orderBy("case_insensitive_value").show(10, truncate=False)


Rows with leading/trailing whitespace by text column:
-RECORD 0-------------
 order_id       | 0   
 customer_id    | 0   
 customer_name  | 3   
 category       | 0   
 city           | 3   
 state          | 0   
 order_status   | 0   
 product        | 0   
 payment_method | 0   

Rows requiring obvious title-case normalization:
-RECORD 0------------
 customer_name | 0   
 category      | 6   
 city          | 6   
 state         | 0   
 order_status  | 0   

Text groups with multiple capitalizations:
customer_name - inconsistent groups: 0
+----------------------+-------------+-----------------+
|case_insensitive_value|variant_count|observed_variants|
+----------------------+-------------+-----------------+
+----------------------+-------------+-----------------+

category - inconsistent groups: 4
+----------------------+-------------+--------------------------+
|case_insensitive_value|variant_count|observed_variants         |
+----------------------+-------------+------------------

### Numeric and date rules

- Quantity must be a positive whole number.
- Unit price must be a positive finite number.
- Discount percentage must be finite and between 0 and 100 inclusive.
- Accepted date formats are `yyyy-MM-dd`, `yyyy/MM/dd`, `MM-dd-yyyy`, and `dd/MM/yyyy`.
  These cover the source formats; slash dates with the year last are interpreted day-first.
  Impossible dates (for example, February 31) and unparseable text become null.
- Dates are validated by Spark's corrected parser with `try_to_timestamp`, which returns
  null on invalid input. No future-date filter or arbitrary outlier cutoff is applied.


In [6]:
numeric_columns = ["quantity", "unit_price", "discount_percent"]

def number(c):
    return F.expr(f"try_cast(`{c}` as double)")

def finite(x):
    return x.isNotNull() & ~F.isnan(x) & (F.abs(x) < F.lit(float("inf")))

def valid_quantity(x):
    return finite(x) & (x > 0) & (x == F.floor(x)) & (x <= 2147483647)

def valid_price(x):
    return finite(x) & (x > 0)

def valid_discount(x):
    return finite(x) & x.between(0, 100)

numeric_rules = {"quantity": valid_quantity, "unit_price": valid_price,
                 "discount_percent": valid_discount}
invalid_numeric_counts = trimmed_raw.agg(*[
    count_true(F.col(c).isNotNull() & ~F.coalesce(rule(number(c)), F.lit(False))).alias(c)
    for c, rule in numeric_rules.items()
])
print("Invalid numeric values, excluding missing fields:")
invalid_numeric_counts.show(truncate=False, vertical=True)
for c, rule in numeric_rules.items():
    print("Invalid", c, "values and their frequencies:")
    (trimmed_raw.filter(F.col(c).isNotNull() & ~F.coalesce(rule(number(c)), F.lit(False)))
     .groupBy(c).count().orderBy(c).show(truncate=False))

DATE_FORMATS = ["yyyy-MM-dd", "yyyy/MM/dd", "MM-dd-yyyy", "dd/MM/yyyy"]
def parsed_date(c):
    return F.coalesce(*[F.try_to_timestamp(F.col(c), F.lit(fmt)).cast("date")
                        for fmt in DATE_FORMATS])

date_inspection = trimmed_raw.withColumn("parsed_order_date", parsed_date("order_date"))
date_counts = date_inspection.agg(
    count_true(F.col("order_date").isNull()).alias("missing_dates"),
    count_true(F.col("order_date").isNotNull() & F.col("parsed_order_date").isNull())
        .alias("invalid_dates"),
    count_true(F.col("parsed_order_date").isNotNull() &
               (F.col("order_date") != F.date_format("parsed_order_date", "yyyy-MM-dd")))
        .alias("valid_dates_needing_format_change"))
date_counts.show(truncate=False)
(date_inspection.filter(F.col("order_date").isNotNull() &
                        (F.col("parsed_order_date").isNull() |
                         (F.col("order_date") != F.date_format("parsed_order_date", "yyyy-MM-dd"))))
 .groupBy("order_date", "parsed_order_date").count().orderBy("order_date")
 .show(truncate=False))


Invalid numeric values, excluding missing fields:
-RECORD 0---------------
 quantity         | 12  
 unit_price       | 8   
 discount_percent | 8   

Invalid quantity values and their frequencies:
+--------+-----+
|quantity|count|
+--------+-----+
|-1      |1    |
|-2      |5    |
|0       |6    |
+--------+-----+

Invalid unit_price values and their frequencies:
+----------+-----+
|unit_price|count|
+----------+-----+
|-250      |1    |
|-500      |3    |
|0         |4    |
+----------+-----+

Invalid discount_percent values and their frequencies:
+----------------+-----+
|discount_percent|count|
+----------------+-----+
|-5              |4    |
|120             |4    |
+----------------+-----+

+-------------+-------------+---------------------------------+
|missing_dates|invalid_dates|valid_dates_needing_format_change|
+-------------+-------------+---------------------------------+
|0            |4            |2                                |
+-------------+-------------+--------

## 5. Basic preprocessing for Silver

1. Remove **exact original duplicates only**, before normalization.
2. Trim fields; convert blanks to null. Fill missing descriptive text with `Unknown`.
   Keep missing identifiers null rather than inventing customer or order IDs.
3. Apply title case to names, categories, cities, states, and statuses. For product and
   payment labels, use the most frequent trimmed spelling within each case-insensitive
   group (lexicographic tie-break). This retains labels such as `UPI`, `T-Shirt`, and `500g`.
4. Convert invalid or missing numeric values to null without dropping their rows or
   inventing sales amounts. Quantity becomes integer; price and discount become double.
5. Parse dates into Spark `DateType`; export valid dates as `yyyy-MM-dd` and invalid dates
   as empty CSV fields. Keep cancelled/returned orders and plausible unusual values.

Null numbers and dates explicitly mean *unknown/unusable*. They must not be interpreted
as zero. This approach retains useful information without claiming perfect cleanliness.


In [7]:
clean = trim_and_null(deduplicated)
for c in title_columns:
    clean = clean.withColumn(c, F.initcap(F.col(c)))

for c in modal_columns:
    frequencies = (trim_and_null(deduplicated).filter(F.col(c).isNotNull())
                   .groupBy(F.lower(F.col(c)).alias("_key"), F.col(c).alias("_label"))
                   .count())
    ranked = frequencies.withColumn(
        "_rank", F.row_number().over(Window.partitionBy("_key")
                                     .orderBy(F.desc("count"), F.asc("_label"))))
    canonical = ranked.filter(F.col("_rank") == 1).select("_key", "_label")
    clean = (clean.withColumn("_key", F.lower(F.col(c)))
             .join(F.broadcast(canonical), "_key", "left")
             .withColumn(c, F.col("_label")).drop("_key", "_label"))

clean = clean.fillna("Unknown", subset=title_columns + modal_columns)
for c, rule in numeric_rules.items():
    x = number(c)
    clean = clean.withColumn(c, F.when(rule(x), x).otherwise(F.lit(None).cast("double")))
clean = clean.withColumn("quantity", F.col("quantity").cast("integer"))
clean = clean.withColumn("order_date", parsed_date("order_date"))
silver_df = clean.select(*expected_columns).cache()
after_count = silver_df.count()
silver_df.printSchema()
silver_df.orderBy("order_id").show(8, truncate=False)
print("Before:", before_count, "| After:", after_count, "| Exact duplicates removed:", duplicate_count)
assert after_count == after_dedup_count == before_count - duplicate_count


root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = false)
 |-- product: string (nullable = false)
 |-- category: string (nullable = false)
 |-- quantity: integer (nullable = true)
 |-- unit_price: double (nullable = true)
 |-- discount_percent: double (nullable = true)
 |-- payment_method: string (nullable = false)
 |-- city: string (nullable = false)
 |-- state: string (nullable = false)
 |-- order_date: date (nullable = true)
 |-- order_status: string (nullable = false)

+--------+-----------+---------------+-------------+---------+--------+----------+----------------+----------------+----------+--------------+----------+------------+
|order_id|customer_id|customer_name  |product      |category |quantity|unit_price|discount_percent|payment_method  |city      |state         |order_date|order_status|
+--------+-----------+---------------+-------------+---------+--------+----------+----------------+-------------

## 6. Save Silver and read it back with Spark

Spark writes CSV to the required directory. Because this is a small assignment dataset,
`coalesce(1)` produces one data part. A byte-for-byte copy of that part is saved as
`silver/sales_silver.csv` for GitHub submission; it contains the **complete** Silver data.
This single-part choice would not be appropriate for a large production data lake.


In [8]:
(silver_df.orderBy("order_id").coalesce(1).write.mode("overwrite")
 .option("header", True).option("dateFormat", "yyyy-MM-dd")
 .option("nullValue", "").csv(str(SILVER.resolve())))
parts = list(SILVER.glob("part-*.csv"))
assert len(parts) == 1, "Expected one complete Spark CSV part."
shutil.copyfile(parts[0], SUBMISSION_CSV)

def read_silver(path):
    return (spark.read.schema(silver_df.schema).option("header", True)
            .option("mode", "FAILFAST").option("dateFormat", "yyyy-MM-dd")
            .csv(str(path.resolve())))

silver_back = read_silver(SILVER)
submission_back = read_silver(SUBMISSION_CSV)
print("Silver read-back record count:", silver_back.count())
silver_back.orderBy("order_id").show(8, truncate=False)
print("Complete submission CSV:", SUBMISSION_CSV)


Silver read-back record count: 990
+--------+-----------+---------------+-------------+---------+--------+----------+----------------+----------------+----------+--------------+----------+------------+
|order_id|customer_id|customer_name  |product      |category |quantity|unit_price|discount_percent|payment_method  |city      |state         |order_date|order_status|
+--------+-----------+---------------+-------------+---------+--------+----------+----------------+----------------+----------+--------------+----------+------------+
|100001  |C0256      |Keerthi Mishra |Saree        |Clothing |2       |6710.0    |20.0            |Credit Card     |Mumbai    |Maharashtra   |2026-01-01|Delivered   |
|100002  |C0042      |Arjun Singh    |T-Shirt      |Clothing |6       |680.0     |5.0             |Debit Card      |Chennai   |Tamil Nadu    |2025-08-03|Delivered   |
|100003  |C0348      |Keerthi Rao    |Hoodie       |Clothing |6       |3080.0    |12.0            |UPI             |Pune      |Mah

## 7. Verify preservation, validation, and output contents


In [9]:
assert hashlib.sha256(BRONZE.read_bytes()).hexdigest() == source_hash
assert silver_back.count() == after_count
assert submission_back.count() == after_count
for restored in [silver_back, submission_back]:
    assert restored.exceptAll(silver_df).count() == 0
    assert silver_df.exceptAll(restored).count() == 0

for c, rule in numeric_rules.items():
    assert silver_back.filter(F.col(c).isNotNull() & ~rule(F.col(c))).count() == 0
for c in text_columns:
    assert silver_back.filter(F.col(c).isNotNull() &
                              (F.col(c) != F.trim(F.col(c)))).count() == 0
for c in title_columns + modal_columns:
    assert silver_back.filter(missing(F.col(c))).count() == 0
for c in title_columns:
    assert silver_back.filter(F.col(c) != F.initcap(F.col(c))).count() == 0
for c in modal_columns:
    assert (silver_back.groupBy(F.lower(F.col(c)))
            .agg(F.countDistinct(c).alias("variants"))
            .filter(F.col("variants") > 1).count()) == 0

print("Remaining nulls per Silver column (intentional for unknown numeric/date values):")
silver_back.agg(*[count_true(F.col(c).isNull()).alias(c) for c in silver_back.columns])    .show(truncate=False, vertical=True)
print("All checks passed: Bronze unchanged; Silver round-trips exactly; row retention and value rules hold.")

summary = {"student": "Abhay Prasad", "roll_number": "LIT2023006",
           "spark_version": spark.version, "source_sha256": source_hash,
           "rows_before": before_count, "exact_duplicates_removed": duplicate_count,
           "rows_after": after_count,
           "bronze_missing_or_blank": missing_counts.first().asDict(),
           "bronze_invalid_numeric": invalid_numeric_counts.first().asDict(),
           "bronze_date_discrepancies": date_counts.first().asDict(),
           "verification": "passed"}
print(json.dumps(summary, indent=2))
(BASE / "verification_summary.json").write_text(json.dumps(summary, indent=2) + "\n", encoding="utf-8")


Remaining nulls per Silver column (intentional for unknown numeric/date values):
-RECORD 0---------------
 order_id         | 0   
 customer_id      | 0   
 customer_name    | 0   
 product          | 0   
 category         | 0   
 quantity         | 12  
 unit_price       | 8   
 discount_percent | 8   
 payment_method   | 0   
 city             | 0   
 state            | 0   
 order_date       | 4   
 order_status     | 0   

All checks passed: Bronze unchanged; Silver round-trips exactly; row retention and value rules hold.
{
  "student": "Abhay Prasad",
  "roll_number": "LIT2023006",
  "spark_version": "4.0.4",
  "source_sha256": "1b2c7e0acea6250c992b890975ed0947b5bececa6a2880b707e4523e0ee22eb8",
  "rows_before": 1000,
  "exact_duplicates_removed": 10,
  "rows_after": 990,
  "bronze_missing_or_blank": {
    "order_id": 0,
    "customer_id": 0,
    "customer_name": 10,
    "product": 0,
    "category": 0,
    "quantity": 0,
    "unit_price": 0,
    "discount_percent": 0,
    "paymen

820

## 8. Submission summary

The workflow removed exact duplicate rows, handled missing descriptive text, trimmed
spaces, standardized obvious capitalization, validated quantity/price/discount, and
normalized valid dates. Invalid numeric/date values remain explicitly null so their other
useful fields are retained. The displayed counts above are calculated by Spark.

Submit this executed notebook as `Data_Lake_Assignment.ipynb`, the generated
`silver/sales_silver.csv`, the unchanged `bronze/sales.csv`, and the local README inside
`submissions/LIT2023006_Abhay_Prasad/`. No changes belong in the class README, dataset
folder, or another student's folder. In Colab, download the notebook with
**File > Download > Download .ipynb**, and download the Silver CSV using the Files panel.


In [10]:
raw.unpersist()
deduplicated.unpersist()
silver_df.unpersist()
spark.stop()
print("Spark session stopped. The notebook can be run again from the first cell.")


Spark session stopped. The notebook can be run again from the first cell.
